In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost):
        super(VectorQuantizer, self).__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost
        # Codebook (Bảng từ điển các vector ẩn)
        self.embedding = nn.Embedding(self.num_embeddings, self.embedding_dim)
        self.embedding.weight.data.uniform_(-1/self.num_embeddings, 1/self.num_embeddings)
    def forward(self, inputs):
        # inputs: [B, C, H, W] -> chuyển về [B, H, W, C] phù hợp để tìm khoảng cách
        inputs = inputs.permute(0, 2, 3, 1).contiguous()
        input_shape = inputs.shape
        
        flat_input = inputs.view(-1, self.embedding_dim)
        
        # Tính khoảng cách L2 giữa inputs và các vector trong codebook
        # d = (x-y)^2 = x^2 + y^2 - 2xy
        distances = (torch.sum(flat_input**2, dim=1, keepdim=True) 
                    + torch.sum(self.embedding.weight**2, dim=1)
                    - 2 * torch.matmul(flat_input, self.embedding.weight.t()))
            
        # Tìm index của vector gần nhất
        encoding_indices = torch.argmin(distances, dim=1).unsqueeze(1)
        encodings = torch.zeros(encoding_indices.shape[0], self.num_embeddings, device=inputs.device)
        encodings.scatter_(1, encoding_indices, 1)
        
        # Lượng tử hóa (Quantize)
        quantized = torch.matmul(encodings, self.embedding.weight).view(input_shape)
        
        # Loss functions
        # 1. e_latent_loss: Cập nhật codebook
        # 2. q_latent_loss: Ép encoder tạo ra vector gần với codebook (commitment loss)
        e_latent_loss = F.mse_loss(quantized.detach(), inputs)
        q_latent_loss = F.mse_loss(quantized, inputs.detach())
        loss = e_latent_loss + self.commitment_cost * q_latent_loss
        
        # Thủ thuật Straight Through Estimator để Backprop qua bước Argmin (không đạo hàm được)
        quantized = inputs + (quantized - inputs).detach()
        
        return loss, quantized.permute(0, 3, 1, 2).contiguous(), encodings

In [ ]:
class Encoder(nn.Module):
    def __init__(self, in_channels, latent_dim):
        super(Encoder, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, latent_dim, kernel_size=1)
        )
    def forward(self, x):
        return self.net(x)
# 3. Decoder
class Decoder(nn.Module):
    def __init__(self, latent_dim, out_channels):
        super(Decoder, self).__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.Conv2d(32, out_channels, kernel_size=3, padding=1),
            nn.Sigmoid() # MNIST giá trị từ 0-1
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
class VQVAE(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost):
        super(VQVAE, self).__init__()
        self.encoder = Encoder(1, embedding_dim) # 1 channel cho MNIST
        self.vq = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
        self.decoder = Decoder(embedding_dim, 1)
    def forward(self, x):
        z = self.encoder(x)
        vq_loss, quantized, _ = self.vq(z)
        x_recon = self.decoder(quantized)
        return vq_loss, x_recon


In [ ]:
def train():
    # Hyperparameters
    batch_size = 128
    num_embeddings = 64
    embedding_dim = 16
    commitment_cost = 0.25
    learning_rate = 1e-3
    num_epochs = 20
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Data Loading
    transform = transforms.Compose([transforms.ToTensor()])
    train_loader = DataLoader(datasets.MNIST('./data', train=True, download=True, transform=transform), 
                              batch_size=batch_size, shuffle=True)
    model = VQVAE(num_embeddings, embedding_dim, commitment_cost).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    model.train()
    for epoch in range(num_epochs):
        total_recon_loss = 0
        total_vq_loss = 0
        
        for batch_idx, (data, _) in enumerate(train_loader):
            data = data.to(device)
            optimizer.zero_grad()
            
            vq_loss, data_recon = model(data)
            
            # Reconstruction Loss
            recon_loss = F.mse_loss(data_recon, data)
            
            loss = recon_loss + vq_loss
            loss.backward()
            
            optimizer.step()
            
            total_recon_loss += recon_loss.item()
            total_vq_loss += vq_loss.item()
        print(f"Epoch [{epoch+1}/{num_epochs}] Recon Loss: {total_recon_loss/len(train_loader):.4f} VQ Loss: {total_vq_loss/len(train_loader):.4f}")
if __name__ == "__main__":
    train()